# Realization study: the four-way ordering does not survive
 
Run on 12 September 2026. 100 independent realizations each of BA, WS and ER at
`N = 34`, four attack strategies, five ASR settings, `K = 512` Sobol points per
evaluation. 6,020 evaluations. Karate is a single fixed network and has one
instance by definition.
 
This addresses Reviewer 2 major comments 1 and 3.
 
---
 
## 1. The committed code reproduces the submitted means exactly
 
All sixteen RASR values at ASR = 100% match the submitted Table 6 to two
decimals, across all four networks and all four strategies. The implementation
is sound and the submitted means are not in question.
 
## 2. The standard deviations in Table 7 do not reproduce
 
Using the paper's stated settings (seed 42, `K = 512`), the recomputed spreads
are consistently higher than those printed:
 
| Network | ASR | paper sd | recomputed sd |
|---|---|---|---|
| Karate | 80% | 13.02 | 13.70 |
| Karate | 60% | 16.29 | 17.92 |
| Scale-Free | 60% | 18.45 | 19.16 |
| Small-World | 80% | 7.82 | 8.60 |
| Random | 50% | 7.54 | 8.63 |
 
Sweeping the Sobol scramble seed over six values gives 13.37 to 13.70 for the
first row, and sweeping `K` over 128 to 1024 gives 13.64 to 14.10. The printed
13.02 lies outside both ranges, so it did not come from this code at these
settings. The means are stable across all of those variations, which is why
they still match.
 
The qualitative claim built on that column survives: hub-dominated networks
still show roughly twice the spread of homogeneous ones. Only the exact figures
change. Table 7 has to be recomputed either way, since it is being replaced by
realization-level statistics.
 
## 3. The four-way ordering is an artifact of one lucky draw
 
Fraction of 100 realizations in which the claimed ordering holds:
 
| ASR | BA < Karate < WS < ER | {BA, Karate} both below {WS, ER} |
|---|---|---|
| 100% | **0%** | 100% |
| 80% | 8% | 100% |
| 60% | 11% | 98% |
| 50% | 6% | 97% |
| random | 31% | 91% |
 
Two independent things break it.
 
**BA and Karate swap.** The submitted BA instance scored 12.27, just below
Karate's 12.86. Across realizations BA averages 16.80 (95% CI 16.45 to 17.16)
and only 1% of draws fall below Karate. The usual relationship is the reverse of
what was claimed.
 
**WS and ER are not separable.** 24.76 versus 25.14 at ASR = 100%,
Mann-Whitney p = 0.21, and WS comes out lower in 55% of realizations. This is a
coin flip, not an ordering.
 
## 4. Why the single draw was misleading
 
The seed-42 BA graph sits at the **97th percentile** of realizations for degree
heterogeneity and the **0th percentile** for RASR. Out of 100 draws it was the
single most hub-dominated and most fragile graph. The seed-42 ER graph sits at
the 87th percentile for RASR, so unusually robust. The claimed ordering needed
both of those at once.
 
For comparison, the WS instance was ordinary: 50th percentile.
 
## 5. What does survive, and it is strong
 
- **Two groups, cleanly separated.** Hub-dominated ({BA, Karate}) below
  homogeneous ({WS, ER}) in 100% of realizations at ASR = 100% and 91% or more
  everywhere else.
- **BA versus WS and BA versus ER**: p around 3e-34, and BA is lower in 100% of
  realizations.
- **Heterogeneity predicts fragility.** Spearman rho = -0.712 (p = 9e-48,
  n = 301 network instances) between `k_max / <k>` and RASR at ASR = 100%. The
  mechanism the paper argues for holds across every instance generated, not just
  the four that were reported.
## 6. Consequence for the paper
 
The defensible claim is a two-group result with a continuous mechanism:
degree heterogeneity governs robustness under probabilistic attack, hub-dominated
classes are systematically more fragile than homogeneous ones, and the finer
ordering within each group is not resolvable at this size. That is a narrower
headline than the submitted one, but it is supported by 100 realizations and a
significance test rather than by one graph per model, which is exactly what
Reviewer 2 asked for.
 
The abstract, contribution 2, Section 5.4.4, Table 9, Section 6.1 and the
conclusion all state the four-way ordering and will need rewriting around the
two-group finding.
 
## 7. Also worth noting for Reviewer 2 minor comment 3
 
The heterogeneous ASR scenario is highly sensitive to the particular draw. The
submitted paper reports 28.24 for Karate under random ASR; a different draw of
the same distribution gives 37.09. Across BA realizations the random-ASR values
span 24.3 to 40.6. Reporting a single draw was not safe, and the revision should
average over draws.
 
---
 
## Files
 
- `results_realizations_N34.csv`: one row per (model, params, realization,
  strategy, ASR), 6,020 rows, with graph statistics and both RASR mean and
  spread.
- `runner.py`: the experiment driver. Wraps `phase4_complete.py` unchanged, adds
  per-realization seeding, tidy output and resume.
- `analyse_realizations.py`: produces every number in this document.